In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name","ecommerce","catalog_name")
dbutils.widgets.text("storage_account_name","abiecommerceadlsdev","storage_account_name")
dbutils.widgets.text("container_name","ecomm-raw-data","container_name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

print(catalog_name, storage_account_name, container_name)

In [0]:
adls_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/order_shipments"
bronze_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/bronze/fact_order_shipment"



In [0]:
df_order_shipment = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", bronze_checkpoint_path) \
  .option("cloudFiles.schemaEvolutionMode", "rescue") \
  .option("cloudFiles.inferColumnTypes", "true") \
  .option("rescuedDataColumn", "_rescued_data") \
  .option("cloudFiles.includeExistingFiles", "true")  \
  .option("pathGlobFilter", "*.csv") \
  .load(adls_path) \
  .withColumn("ingest_timestamp", F.current_timestamp()) \
  .withColumn("source_file", F.col("_metadata.file_path")) \
  .writeStream \
  .outputMode("append") \
  .option("checkpointLocation", bronze_checkpoint_path) \
  .trigger(availableNow=True) \
  .toTable(f"{dbutils.widgets.get('catalog_name')}.bronze.brz_order_shipment") \
  .awaitTermination()